# TD3 baseline on Gridworld maze

In [ ]:
from pathlib import Path
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import time
from copy import deepcopy


# Resolve repository src path robustly when running notebook in-place.
repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / 'src').exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / 'src'))

from environments.maze import MazeGridWorld, MazeGoalWrapper
from utils import (TrajectoryReplayBuffer, evaluate_policy, set_seed, build_goal_batch, get_base_env, collect_valid_states_fourrooms,
                   estimate_fisher_diag, extract_fixed_probe_sa_embedding, extract_mean_sa_embedding, extract_sa_batch_for_isotropy,
                   compute_embedding_drift, collect_weight_snapshot)
from visualisations import visualise_embeddings, visualise_q_table_td3, print_goal_embedding_similarity, plot_full_embedding_dashboard_html
from loss_functions import repulsion_loss_to_memory, sigreg_loss, orthogonal_loss, ewc_regulariser_loss, weight_regulariser_loss
from networks import Factorised_TD3_Critic
from agents import Factorised_TD3_Actor
from trainer import td3_train


DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('Using device:', DEVICE)


In [ ]:
MAZE_LAYOUT = [
    [1,1,1,1,1,1,1,1,1,1,1],
    [1,0,0,0,0,1,0,0,0,0,1],
    [1,0,1,1,0,1,0,1,1,0,1],
    [1,0,1,0,0,0,0,0,1,0,1],
    [1,0,1,0,1,1,1,0,1,0,1],
    [1,0,0,0,1,0,0,0,1,0,1],
    [1,1,1,0,1,0,1,1,1,0,1],
    [1,0,0,0,0,0,1,0,0,0,1],
    [1,0,1,1,1,0,1,1,1,0,1],
    [1,0,0,0,1,0,0,0,0,0,1],
    [1,1,1,1,1,1,1,1,1,1,1],
]

def make_env(goal=(9, 9), slip_prob=0.00, max_horizon=1500):
    base = MazeGridWorld(
        maze=MAZE_LAYOUT,
        max_episode_steps=max_horizon,
    )
    env = MazeGoalWrapper(
        base,
        goal_position=goal,
        goal_reward=1.0,
        step_reward=0.0,
        slip_prob=slip_prob,
        reward_mode="simple",
    )
    return env

env = make_env(goal=(9, 9))
obs, info = env.reset()

img = env.unwrapped.render()
goal = env.goal_position

plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.scatter(
    goal[0] * 40 + 20,
    goal[1] * 40 + 20,
    c="lime",
    s=180,
    marker="*",
    edgecolors="black",
)
plt.title(f"Maze with goal at {goal}")
plt.axis("off")
plt.show()

def make_2d_grid_sampler(
    low,
    high,
    grid_size=41,
    fixed_obs_fn=None,
):
    low = np.asarray(low, dtype=np.float32)
    high = np.asarray(high, dtype=np.float32)

    def sampler():
        xs = np.linspace(low[0], high[0], grid_size)
        ys = np.linspace(low[1], high[1], grid_size)
        XX, YY = np.meshgrid(xs, ys)

        pts2d = np.stack([XX.ravel(), YY.ravel()], axis=-1).astype(np.float32)

        if fixed_obs_fn is None:
            obs_batch = pts2d
        else:
            obs_batch = fixed_obs_fn(pts2d)

        meta = {
            "mode": "grid",
            "xs": xs,
            "ys": ys,
            "XX": XX,
            "YY": YY,
            "points_2d": pts2d,
            "grid_size": grid_size,
        }
        return obs_batch, meta

    return sampler

## Training Loop 

In [ ]:
SEEDS = [42]
GOALS = [(9, 9), (7, 7), (3, 5), (3, 9), (5, 5), (3, 6), (9, 1), (1, 1), (7, 4), (9, 8), (9, 7)]
BUFFER_CAPACITY = 100000
LR = float(1e-3)

sa_keywords_local = ["sa_encoder"]
goal_keywords_local = ["goal_encoder"]

overall_results = {
    goal: {
        "eval_returns": [],
        "eval_returns_time": [],
        "min_steps": [],
        "min_time": [],
        "task_embeddings": [],
        "sa_embeddings": [],
        "sa_fixed_probe_embeddings": [],
        "sa_batches_final": [],
    }
    for goal in GOALS
}

for seed in SEEDS:
    print(f"\n================ SEED {seed} ================\n")
    set_seed(seed)

    env = make_env(goal=GOALS[0])
    obs_dim = env.observation_space.shape[0]
    act_dim = env.action_space.shape[0]
    print(f"Observation dim: {obs_dim}, Action dim: {act_dim}")
    env.close()

    goal_dim = len(GOALS[0])

    actor = Factorised_TD3_Actor(obs_dim, act_dim, goal_dim=goal_dim).to(DEVICE)
    actor_tgt = Factorised_TD3_Actor(obs_dim, act_dim, goal_dim=goal_dim).to(DEVICE)

    q1 = Factorised_TD3_Critic(obs_dim, act_dim, goal_dim=goal_dim).to(DEVICE)
    q2 = Factorised_TD3_Critic(obs_dim, act_dim, goal_dim=goal_dim).to(DEVICE)
    q1_tgt = Factorised_TD3_Critic(obs_dim, act_dim, goal_dim=goal_dim).to(DEVICE)
    q2_tgt = Factorised_TD3_Critic(obs_dim, act_dim, goal_dim=goal_dim).to(DEVICE)
    q1_tgt.load_state_dict(q1.state_dict())
    q2_tgt.load_state_dict(q2.state_dict())

    for p in list(actor_tgt.parameters()) + list(q1_tgt.parameters()) + list(q2_tgt.parameters()):
        p.requires_grad_(False)

    seed_task_embedding_memory = []
    seen_goal_labels = []
    weight_history = []

    prev_actor = None
    prev_actor_tgt = None
    prev_q1 = None
    prev_q2 = None
    prev_q1_tgt = None
    prev_q2_tgt = None

    for goal_idx, goal in enumerate(GOALS):
        print(f"\n----- seed={seed}, goal={goal} -----\n")

        if goal_idx == 0:
            actor_curr = deepcopy(actor)
            actor_tgt_curr = deepcopy(actor_tgt)
            q1_curr = deepcopy(q1)
            q2_curr = deepcopy(q2)
            q1_tgt_curr = deepcopy(q1_tgt)
            q2_tgt_curr = deepcopy(q2_tgt)

            actor_tgt_curr.load_state_dict(actor_curr.state_dict())
            q1_tgt_curr.load_state_dict(q1_curr.state_dict())
            q2_tgt_curr.load_state_dict(q2_curr.state_dict())
        else:
            if any(x is None for x in [prev_actor, prev_actor_tgt, prev_q1, prev_q2, prev_q1_tgt, prev_q2_tgt]):
                raise ValueError("Previous TD3 networks are not available for transfer.")

            actor_curr = deepcopy(prev_actor)
            actor_tgt_curr = deepcopy(prev_actor_tgt)
            q1_curr = deepcopy(prev_q1)
            q2_curr = deepcopy(prev_q2)
            q1_tgt_curr = deepcopy(prev_q1_tgt)
            q2_tgt_curr = deepcopy(prev_q2_tgt)

            actor_tgt_curr.load_state_dict(actor_curr.state_dict())
            q1_tgt_curr.load_state_dict(q1_curr.state_dict())
            q2_tgt_curr.load_state_dict(q2_curr.state_dict())

        for p in list(actor_tgt_curr.parameters()) + list(q1_tgt_curr.parameters()) + list(q2_tgt_curr.parameters()):
            p.requires_grad_(False)

        weight_history.append(
            collect_weight_snapshot(
                qnet=q1_curr,
                goal_label=goal,
                stage_label=f"goal_{goal_idx}_before_train_{goal}",
                sa_keywords_local=sa_keywords_local,
                goal_keywords_local=goal_keywords_local,
                max_samples_per_group=40000,
            )
        )

        (   actor_trained,
            actor_tgt_trained,
            q1_main,
            q1_tgt_main,
            q2_trained,
            q2_tgt_trained,
            eval_returns,
            min_steps,
            min_time,
            task_embedding,
            sa_embedding_mean,
            sa_embedding_fixed,
            sa_batch_final,
            replay,
        ) = td3_train(
            seed=seed,
            actor=actor_curr,
            actor_tgt=actor_tgt_curr,
            q1=q1_curr,
            q2=q2_curr,
            q1_tgt=q1_tgt_curr,
            q2_tgt=q2_tgt_curr,
            env=make_env(goal=goal),
            make_env=make_env,
            goal=goal,
            buffer_capacity=BUFFER_CAPACITY,
            total_steps=500000,
            warmup_steps=10000,
            batch_size=256,
            gamma=0.99,
            tau=0.005,
            policy_noise=0.2,
            noise_clip=0.5,
            policy_delay=2,
            expl_noise=0.15,
            lr=3e-4,
            train_freq=1,
            gradient_steps=1,
            eval_every=5000,
            device=DEVICE,
            obs_dim=obs_dim,
            act_dim=act_dim,
        )

        weight_history.append(
            collect_weight_snapshot(
                qnet=q1_main,
                goal_label=goal,
                stage_label=f"goal_{goal_idx}_after_train_{goal}",
                sa_keywords_local=sa_keywords_local,
                goal_keywords_local=goal_keywords_local,
                max_samples_per_group=40000,
            )
        )

        visualise_q_table_td3(
            goal=goal,
            actor=actor_trained,
            q1=q1_main,
            q2=q2_trained,
            eval_returns=eval_returns,
            device=DEVICE,
            make_env=make_env,
        )
        
        visualise_embeddings(
            goal=goal,
            q_network=q1_main,
            actor=actor_trained,
            device=DEVICE,
            make_env=make_env,
            action_mode="actor",
        )

        seed_task_embedding_memory.append(task_embedding)
        seen_goal_labels.append(str(goal))
        print_goal_embedding_similarity(seed_task_embedding_memory, goal_labels=seen_goal_labels)

        overall_results[goal]["eval_returns"].append(eval_returns)
        overall_results[goal]["eval_returns_time"].append(eval_returns_time)
        overall_results[goal]["min_steps"].append(min_steps)
        overall_results[goal]["min_time"].append(min_time)
        overall_results[goal]["task_embeddings"].append(task_embedding)
        overall_results[goal]["sa_embeddings"].append(sa_embedding_mean)
        overall_results[goal]["sa_fixed_probe_embeddings"].append(sa_embedding_fixed)
        overall_results[goal]["sa_batches_final"].append(sa_batch_final)

        prev_actor = deepcopy(actor_trained)
        prev_actor_tgt = deepcopy(actor_tgt_trained)
        prev_q1 = deepcopy(q1_main)
        prev_q2 = deepcopy(q2_trained)
        prev_q1_tgt = deepcopy(q1_tgt_main)
        prev_q2_tgt = deepcopy(q2_tgt_trained)

## Visualisations